# End-to-End Flood Detection Training

This notebook trains an end-to-end model: Clay encoder (frozen) + decoder + classifier.

Unlike the two-stage approach, this processes raw images directly without pre-computing embeddings.

In [1]:
import os
from pathlib import Path

def find_project_root(marker='claymodel'):
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Project root not found")

os.chdir(find_project_root())
print(f"Working directory: {os.getcwd()}")

Working directory: /home/ignacio/Code/Clay-foundation/model


In [2]:
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.cli import LightningArgumentParser, instantiate_class
import lightning.pytorch as L
from claymodel.finetune.flood_detection.end_to_end_flood_classifier import EndToEndFloodClassifier
from claymodel.finetune.flood_detection.end_to_end_gfm_datamodule import EndToEndGFMDataModule
from datetime import datetime

seed_everything(42)

Seed set to 42


42

In [3]:
def train_end_to_end_model(config_path, test_after_training=False):
    objects = ["callbacks", "logger", "plugins"]
    parser = LightningArgumentParser()
    parser.add_lightning_class_args(EndToEndFloodClassifier, "model")
    parser.add_lightning_class_args(EndToEndGFMDataModule, "data")
    parser.add_lightning_class_args(Trainer, "trainer")
    config = parser.parse_path(config_path)
    trainer_config = dict(config["trainer"])
    
    for object_type in objects:
        if object_type in trainer_config and trainer_config[object_type]:
            instantiated = []
            for item_config in trainer_config[object_type]:
                if hasattr(item_config, 'class_path') and hasattr(item_config, 'init_args'):
                    if object_type == "logger":
                        if item_config.class_path == "lightning.pytorch.loggers.CSVLogger":
                            item_config.init_args['version'] = datetime.now().strftime("%Y%m%d_%H%M%S")
                        elif item_config.class_path == "lightning.pytorch.loggers.WandbLogger":
                            item_config.init_args['name'] = "EndToEnd_" + datetime.now().strftime("%Y%m%d_%H%M%S")
                    if object_type == "callbacks" and item_config.class_path == "lightning.pytorch.callbacks.ModelCheckpoint":
                        item_config.init_args['dirpath'] = os.path.join(item_config.init_args['dirpath'], datetime.now().strftime("%Y%m%d_%H%M%S"))
                    item = instantiate_class((), item_config)
                    instantiated.append(item)
                elif isinstance(item_config, dict) and "class_path" in item_config:
                    item = instantiate_class((), item_config)
                    instantiated.append(item)
                else:
                    instantiated.append(item_config)
            trainer_config[object_type] = instantiated
    
    model = EndToEndFloodClassifier(**config["model"])
    datamodule = EndToEndGFMDataModule(**config["data"])
    trainer = Trainer(**trainer_config)
    
    try:
        model_config = dict(config["model"])
        for logger in trainer.loggers:
            if isinstance(logger, L.pytorch.loggers.WandbLogger):
                run = logger.experiment
                if run is not None:
                    run.config.update({"model": model_config}, allow_val_change=True)
            elif isinstance(logger, L.pytorch.loggers.CSVLogger):
                logger.log_hyperparams({"model": model_config})
    except Exception as e:
        print(f"Warning: {e}")
    
    result = trainer.fit(model, datamodule)
    if test_after_training:
        result = trainer.test(model, datamodule)
    print("Results:", result)
    return model, datamodule, trainer, result

## Train the Model

In [4]:
CONFIG_PATH = "configs/train_end_to_end_flood_detection.yaml"
model, datamodule, trainer, result = train_end_to_end_model(CONFIG_PATH, test_after_training=True)

🔄 Loading Clay encoder from: checkpoints/clay-v1.5.ckpt
🔒 Clay encoder frozen
🏗️  EndToEndFloodClassifier initialized:
   Input: Raw images (224, 224)
   Encoder: Clay (frozen=True)
   Decoder: 512D hidden, ASPP=True, Residual=True
   Fusion: early_concat_diff
   Output: Binary flood segmentation


/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/accelerator_connector.py:508: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name              | Type                           | Params | Mode 
-----------------------------------------------------------------------------
0 | clay_encoder      | Encoder                        | 311 M  | eval 
1 | segmentation_head | TemporalFusionSegmentationHead | 3

📊 Dataset initialized: 100 paired samples from data/GFM/tif/train_split_balanced_true/chips
📊 Dataset initialized: 35 paired samples from data/GFM/tif/val_split_balanced_false/chips


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:527: Found 287 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
import lightning as L


def count_confusion_pixels(
    model: L.LightningModule,
    datamodule: L.LightningDataModule,
    trainer: L.Trainer,
    split: str = "test",
    print_results: bool = False
):
    """
    Count all pixels across entire dataset split for confusion matrix analysis.
    
    Counts:
    - Exclusion pixels (ignored areas)
    - TN (True Negative): correctly predicted non-flood
    - TP (True Positive): correctly predicted flood
    - FN (False Negative): missed flood
    - FP (False Positive): incorrectly predicted flood
    
    Args:
        model: Lightning model
        datamodule: Lightning datamodule with the dataset
        trainer: Lightning trainer
        split: Dataset split ('train', 'val', or 'test')
        print_results: Whether to print detailed results
    
    Returns:
        dict: Pixel counts for each category and computed metrics
    """
    # Get appropriate dataloader
    if split == "train":
        dataloader = datamodule.train_dataloader()
    elif split == "val":
        dataloader = datamodule.val_dataloader()
    elif split == "test":
        dataloader = datamodule.test_dataloader()
    else:
        raise ValueError("split must be 'train', 'val' or 'test'")

    # Initialize counters
    counts = {
        "exclusion": 0,
        "TN": 0,
        "TP": 0,
        "FN": 0,
        "FP": 0,
        "total_pixels": 0,
        "valid_pixels": 0,  # pixels not in exclusion zone
    }

    # Set model to eval mode
    model.eval()
    
    # Move model to appropriate device
    device = trainer.strategy.root_device if hasattr(trainer.strategy, 'root_device') else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    print(f"Processing {len(dataloader)} batches from {split} split...")

    # Process all batches
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Processing {split} chips"):
            # Get batch data
#            pre_embedding = batch["pre_embedding"].to(device)
#            post_embedding = batch["post_embedding"].to(device)
            pre_image = batch["pre_image"].to(device)
            post_image = batch["post_image"].to(device)
            labels = batch["label"]  # Keep on CPU for efficiency
            
            # Get masks if available
            if "ignore_mask" in batch:
                ignore_mask = batch["ignore_mask"]
            else:
                # Construct ignore mask from exclusion and water masks if provided
                excl_mask = batch.get("exclusion_mask", torch.zeros_like(labels, dtype=torch.bool))
                water_mask = batch.get("water_mask", torch.zeros_like(labels, dtype=torch.bool))
                ignore_mask = excl_mask | water_mask

            # Forward pass
            batch_input = {
                #"pre_embedding": pre_embedding,
                #"post_embedding": post_embedding,
                "pre_image": pre_image,
                "post_image": post_image,
            }
            logits = model(batch_input)
            
            # Get predictions
            probs = torch.sigmoid(logits).squeeze(1)  # [B, H, W]
            pred_mask = (probs > 0.5).cpu().numpy().astype(np.uint8)
            
            # Convert to numpy for counting
            labels = labels.cpu().numpy().astype(np.uint8)
            ignore_mask = ignore_mask.cpu().numpy().astype(bool)

            # Process each item in batch
            batch_size = labels.shape[0]
            for i in range(batch_size):
                label = labels[i]
                pred = pred_mask[i]
                ignore = ignore_mask[i]
                
                # Count pixels
                total_pixels = label.size
                counts["total_pixels"] += total_pixels
                
                # Count exclusion pixels
                exclusion_pixels = np.sum(ignore)
                counts["exclusion"] += exclusion_pixels
                
                # Create mask for valid pixels (not excluded)
                valid_mask = ~ignore
                counts["valid_pixels"] += np.sum(valid_mask)
                
                # Count confusion matrix elements (only on valid pixels)
                counts["TN"] += np.sum((label == 0) & (pred == 0) & valid_mask)
                counts["TP"] += np.sum((label == 1) & (pred == 1) & valid_mask)
                counts["FN"] += np.sum((label == 1) & (pred == 0) & valid_mask)
                counts["FP"] += np.sum((label == 0) & (pred == 1) & valid_mask)

    # Calculate metrics
    confusion_sum = counts['TN'] + counts['TP'] + counts['FN'] + counts['FP']
    
    accuracy = (counts['TP'] + counts['TN']) / counts['valid_pixels'] if counts['valid_pixels'] > 0 else 0
    precision = counts['TP'] / (counts['TP'] + counts['FP']) if (counts['TP'] + counts['FP']) > 0 else 0
    recall = counts['TP'] / (counts['TP'] + counts['FN']) if (counts['TP'] + counts['FN']) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    iou = counts['TP'] / (counts['TP'] + counts['FP'] + counts['FN']) if (counts['TP'] + counts['FP'] + counts['FN']) > 0 else 0
    
    # Add metrics to return dict
    counts['metrics'] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
    }
    
    total_flood = counts['TP'] + counts['FN']
    total_no_flood = counts['TN'] + counts['FP']
    
    if print_results:
        print_confusion_results(counts, split, len(dataloader), confusion_sum, 
                              total_flood, total_no_flood)
    
    return counts


def print_confusion_results(counts, split, num_batches, confusion_sum, 
                           total_flood, total_no_flood):
    """Helper function to print formatted results."""
    metrics = counts['metrics']
    
    print("\n" + "="*60)
    print(f"CONFUSION MATRIX PIXEL COUNTS - {split.upper()} SPLIT")
    print("="*60)
    print(f"\nTotal batches processed: {num_batches}")
    print(f"Total pixels: {counts['total_pixels']:,}")
    print(f"\nExclusion pixels: {counts['exclusion']:,} ({100*counts['exclusion']/counts['total_pixels']:.2f}%)")
    print(f"Valid pixels: {counts['valid_pixels']:,} ({100*counts['valid_pixels']/counts['total_pixels']:.2f}%)")
    
    print(f"\n{'─'*60}")
    print("CONFUSION MATRIX (Valid Pixels Only):")
    print(f"{'─'*60}")
    print(f"True Negatives  (TN): {counts['TN']:>12,} ({100*counts['TN']/counts['valid_pixels']:.2f}%)")
    print(f"True Positives  (TP): {counts['TP']:>12,} ({100*counts['TP']/counts['valid_pixels']:.2f}%)")
    print(f"False Negatives (FN): {counts['FN']:>12,} ({100*counts['FN']/counts['valid_pixels']:.2f}%)")
    print(f"False Positives (FP): {counts['FP']:>12,} ({100*counts['FP']/counts['valid_pixels']:.2f}%)")
    
    print(f"\nConfusion matrix sum: {confusion_sum:,}")
    print(f"Valid pixels:         {counts['valid_pixels']:,}")
    print(f"Match: {confusion_sum == counts['valid_pixels']}")
    
    print(f"\n{'─'*60}")
    print("METRICS:")
    print(f"{'─'*60}")
    print(f"Overall Accuracy: {metrics['accuracy']:.4f} ({100*metrics['accuracy']:.2f}%)")
    print(f"Precision:        {metrics['precision']:.4f} ({100*metrics['precision']:.2f}%)")
    print(f"Recall:           {metrics['recall']:.4f} ({100*metrics['recall']:.2f}%)")
    print(f"F1 Score:         {metrics['f1']:.4f} ({100*metrics['f1']:.2f}%)")
    print(f"IoU (Jaccard):    {metrics['iou']:.4f} ({100*metrics['iou']:.2f}%)")
    
    print(f"\n{'─'*60}")
    print("CLASS DISTRIBUTION (Valid Pixels):")
    print(f"{'─'*60}")
    print(f"Flood pixels:     {total_flood:>12,} ({100*total_flood/counts['valid_pixels']:.2f}%)")
    print(f"No-flood pixels:  {total_no_flood:>12,} ({100*total_no_flood/counts['valid_pixels']:.2f}%)")
    
    print("="*60 + "\n")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

for split in ["val", "test"]:

    results=count_confusion_pixels(model, datamodule, trainer, split=split)
    
    cm = np.zeros((2,2), dtype=np.int64)
    D = cm[0,0] = results["TN"]
    A = cm[1,1] = results["TP"]
    B = cm[1,0] = results["FN"]
    C = cm[0,1] = results["FP"]
    cm_normalized = cm / results["valid_pixels"]

    disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=['No Flood', 'Flood'])
    disp.plot(cmap='Blues', xticks_rotation=45)
    plt.title('Normalized Confusion Matrix')
    plt.show()
    
    CSI= A/(A+B+C)
    OA=(A+D)/(A+B+C+D)
    C_e=B/(A+B)
    O_e=C/(A+C)
    print(f"CSI:{CSI:.3f} OA:{OA:.3f} C_e:{C_e:.3f} O_e:{O_e:.3f} total_pixels:{cm.sum()}")

In [ ]:
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import lightning as L


def plot_random_n_chips_overlapped(
    model: L.LightningModule,
    datamodule: L.LightningDataModule,
    trainer: L.Trainer,
    n: int = 5,
    split: str = "test"
):
    """
    Visualize random chips for the GFM setup using paired pre/post embeddings and binary flood labels.

    Shows: pre SIG0, post SIG0, and confusion matrix overlay where:
    - TN (True Negative): Black (K)
    - TP (True Positive): White (W)
    - FP (False Positive): Green (G)
    - FN (False Negative): Blue (B)
    - Exclusion: Red (R)
    
    Args:
        model: Lightning model
        datamodule: Lightning datamodule with the dataset
        trainer: Lightning trainer
        n: Number of random chips to visualize
        split: Dataset split ('train', 'val', or 'test')
    """
    # Get appropriate dataloader
    if split == "train":
        dataloader = datamodule.train_dataloader()
    elif split == "val":
        dataloader = datamodule.val_dataloader()
    elif split == "test":
        dataloader = datamodule.test_dataloader()
    else:
        raise ValueError("split must be 'train', 'val' or 'test'")

    # Collect all samples from dataloader
    all_samples = []

    for batch in dataloader:
        batch_size = batch["label"].shape[0]
        for i in range(batch_size):
            sample = {key: val[i] for key, val in batch.items()}
            all_samples.append(sample)
    
    if len(all_samples) == 0:
        raise RuntimeError(f"No samples found in {split} split")
    
    if n > len(all_samples):
        raise ValueError(
            f"n ({n}) is greater than the number of available samples ({len(all_samples)})"
        )

    # Select random samples
    selected = random.sample(all_samples, n)
    
    # Set model to eval mode
    model.eval()
    
    # Move model to appropriate device
    device = trainer.strategy.root_device if hasattr(trainer.strategy, 'root_device') else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # Create figure with 3 columns: Pre, Post, Overlay
    _, axs = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axs = np.expand_dims(axs, axis=0)

    # Define colormap: 0=TN(Black), 1=TP(White), 2=FP(Green), 3=FN(Blue), 4=Exclusion(Red)
    colors = ['black', 'white', 'green', 'blue', 'red']
    cmap = ListedColormap(colors)

    for row_idx, sample in enumerate(selected):
        # Get sample data
        #pre_embedding = sample["pre_embedding"].unsqueeze(0).to(device)
        #post_embedding = sample["post_embedding"].unsqueeze(0).to(device)
        pre_image = sample["pre_image"].unsqueeze(0).to(device)
        post_image = sample["post_image"].unsqueeze(0).to(device)
        label = sample["label"].cpu().numpy().astype(np.uint8)
        
        # Get ignore mask
        if "ignore_mask" in sample:
            ignore_mask = sample["ignore_mask"].cpu().numpy().astype(bool)
        else:
            excl_mask = sample.get("exclusion_mask", torch.zeros_like(sample["label"], dtype=torch.bool))
            water_mask = sample.get("water_mask", torch.zeros_like(sample["label"], dtype=torch.bool))
            ignore_mask = (excl_mask | water_mask).cpu().numpy().astype(bool)

        # Extract first channel of images for visualization
        pre_img = pre_image[0, 0].cpu().numpy()  # First channel (e.g., SIG0)
        post_img = post_image[0, 0].cpu().numpy()

        # Predict
        with torch.no_grad():
            batch = {
                #"pre_embedding": pre_embedding,
                #"post_embedding": post_embedding,
                "pre_image": pre_image,
                "post_image": post_image,
            }
            logits = model(batch)
            probs = torch.sigmoid(logits).squeeze(0).squeeze(0).cpu().numpy()
            pred_mask = (probs > 0.5).astype(np.uint8)

        # Create confusion matrix overlay
        # Initialize with TN (0 = black)
        overlay = np.zeros_like(label, dtype=np.uint8)
        
        # True Positives: label=1 AND pred=1 -> 1 (white)
        overlay[(label == 1) & (pred_mask == 1)] = 1
        
        # False Positives: label=0 AND pred=1 -> 2 (green)
        overlay[(label == 0) & (pred_mask == 1)] = 2
        
        # False Negatives: label=1 AND pred=0 -> 3 (blue)
        overlay[(label == 1) & (pred_mask == 0)] = 3
        
        # Exclusion layer overrides everything -> 4 (red)
        overlay[ignore_mask] = 4

        # Plot Pre SIG0
        axs[row_idx, 0].imshow(pre_img, cmap="gray")
        axs[row_idx, 0].set_title("Pre SIG0")
        axs[row_idx, 0].axis("off")

        # Plot Post SIG0
        axs[row_idx, 1].imshow(post_img, cmap="gray")
        axs[row_idx, 1].set_title("Post SIG0")
        axs[row_idx, 1].axis("off")

        # Plot overlay with custom colormap
        axs[row_idx, 2].imshow(overlay, cmap=cmap, vmin=0, vmax=4)
        axs[row_idx, 2].set_title("Overlay (TN:K, TP:W, FP:G, FN:B, Excl:R)")
        axs[row_idx, 2].axis("off")

    plt.tight_layout()
    plt.show()